# 05 – DistilBERT Data Preparation

**Purpose:** Build and verify the data pipelines for future fine-tuning of a `DistilBERT` model.

This notebook covers:
1. Loading the preprocessed training and test splits.
2. Initializing `TransformerTicketDataset` with automatic Hugging Face tokenization.
3. Demonstrating local caching of pre-tokenized features.
4. Constructing a PyTorch `DataLoader` with our custom `DynamicPaddingCollator`.
5. Verifying that batch shapes fluctuate dynamically depending on sequence lengths.

## 0. Setup and Imports

In [ ]:
import sys
from pathlib import Path
from torch.utils.data import DataLoader

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")

## 1. Load Preprocessed Data Splits

We load train and test splits using our data ingestion module.

In [ ]:
from src.data.dataset import load_and_preprocess_dataset

splits = load_and_preprocess_dataset()
train_df = splits["train"]
test_df = splits["test"]

print(f"Train records: {len(train_df)}")
print(f"Test records:  {len(test_df)}")

## 2. Initialize TransformerTicketDataset and Caching

We construct the dataset and cache the features locally to avoid redundant Hugging Face tokenization.

In [ ]:
from src.models.transformer.dataset import TransformerTicketDataset

cache_path = REPO_ROOT / "data" / "transformer_cache" / "train_tokenized.pt"

train_dataset = TransformerTicketDataset(
    texts=train_df["text"].tolist(),
    labels=train_df["label"].tolist(),
    model_name="distilbert-base-uncased",
    max_length=128,
    cache_path=cache_path,
    use_cache=True,
)

print(f"Dataset initialized. Length: {len(train_dataset)}")
print("Sample items retrieved:")
sample = train_dataset[0]
print(f"  - input_ids shape: {sample['input_ids'].shape}")
print(f"  - label:           {sample['label']}")

## 3. Verify Batch Creation & Dynamic Padding Collator

We create a PyTorch DataLoader and verify that dynamic padding keeps sequence dimensions compact.

In [ ]:
from src.models.transformer.collator import DynamicPaddingCollator

# Custom collator using the tokenizer's padding token ID
collator = DynamicPaddingCollator(pad_token_id=train_dataset.tokenizer.pad_token_id)

loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=collator,
)

print("Iterating first 5 batches and printing padded shapes:")
for batch_idx, batch in enumerate(loader):
    if batch_idx >= 5:
        break
    print(f"Batch {batch_idx + 1}:")
    print(f"  - input_ids shape:      {batch['input_ids'].shape}")
    print(f"  - attention_mask shape: {batch['attention_mask'].shape}")
    print(f"  - labels shape:         {batch['labels'].shape}")